In [11]:
# import necessary libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
import shutil
import kagglehub
import os

In [2]:
# Step 1: Download raw .csv data file to data/raw
path = kagglehub.dataset_download("thedevastator/predicting-credit-card-customer-attrition-with-m")
csv_file = os.path.join(path, "BankChurners.csv")
target_folder = "../data/raw/"
os.makedirs(target_folder, exist_ok=True)
shutil.copy(csv_file, target_folder)

'../data/raw/BankChurners.csv'

In [3]:
# Step 2: Load local .csv file into pandas df
raw_data_dir = os.path.join("..", "data", "raw")
filename = "BankChurners.csv"
csv_local = os.path.join(raw_data_dir, filename)

df = pd.read_csv(csv_local)
df.head() # view snippet of data layout

,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,...,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,...,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,0.000093,0.99991
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,...,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,0.000057,0.99994
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,...,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,0.000021,0.99998
3,769911858,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,...,3313.0,2517,796.0,1.405,1171,20,2.333,0.760,0.000134,0.99987
4,709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,...,4716.0,0,4716.0,2.175,816,28,2.500,0.000,0.000022,0.99998


In [4]:
# Step 3: Seperate features (X) and target (Y)
X = df.drop('Attrition_Flag', axis=1)
y = df['Attrition_Flag']

In [5]:
# Step 4: Split into training, validation, and test data
X_train, X_rest, y_train, y_rest = train_test_split(
    X, y, test_size=0.30, random_state=67, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_rest, y_rest, test_size=0.50, random_state=67, stratify=y_rest
)

In [6]:
# Step 5: Ordinal Encoding on all sets

# Find modes for unknown values
print('Most common education level: ' + df['Education_Level'].mode()[0])
print('Most common income category: ' + df['Income_Category'].mode()[0])

# Define education ordering 
edu_map = {
    'Uneducated': 0, 
    'High School': 1, 
    'College': 2, 
    'Graduate': 3, 
    'Post-Graduate': 4, 
    'Doctorate': 5, 
    'Unknown': 3 # replace unknown with mode (graduate)
}

# Define income ordering
income_map = {
    'Less than $40K': 0, 
    '$40K - $60K': 1, 
    '$60K - $80K': 2, 
    '$80K - $120K': 3, 
    '$120K +': 4, 
    'Unknown': 0 # replace unknown with mode (less than $40k)
}

# Apply ordinal encodings
for data in [X_train, X_val, X_test]:
    data['Education_Level'] = data['Education_Level'].map(edu_map)
    data['Income_Category'] = data['Income_Category'].map(income_map)

Most common education level: Graduate
Most common income category: Less than $40K


In [7]:
# Step 6: One-Hot Encoding on all sets

# Uncomment print statements to show unique categories
# print(df['Marital_Status'].unique())
# print(df['Gender'].unique())
# print(df['Card_Category'].unique())

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first').set_output(transform='pandas')
ohe.fit(X_train[['Marital_Status', 'Gender', 'Card_Category']])

# Apply one hot encoding
X_train_ohe = ohe.transform(X_train[['Marital_Status', 'Gender', 'Card_Category']])
X_val_ohe = ohe.transform(X_val[['Marital_Status', 'Gender', 'Card_Category']])
X_test_ohe = ohe.transform(X_test[['Marital_Status', 'Gender', 'Card_Category']])

# Join one-hot-encoded data back to orignal sets
X_train_enc = pd.concat([X_train, X_train_ohe], axis=1).drop(columns=['Marital_Status', 'Gender', 'Card_Category'])
X_val_enc   = pd.concat([X_val, X_val_ohe], axis=1).drop(columns=['Marital_Status', 'Gender', 'Card_Category'])
X_test_enc  = pd.concat([X_test, X_test_ohe], axis=1).drop(columns=['Marital_Status', 'Gender', 'Card_Category'])

In [9]:
# Step 7: SMOTE training data
smote = SMOTE(random_state=67)
X_train_sm, y_train_sm = smote.fit_resample(X_train_enc, y_train)

In [13]:
# Step 8: Standardize data for all sets
scaler = StandardScaler().set_output(transform='pandas')
X_train_final = scaler.fit_transform(X_train_sm)
X_val_final = scaler.transform(X_val_enc)
X_test_final = scaler.transform(X_test_enc)

In [15]:
# Step 9: Export Processed Data
output_path = '../data/processed'
X_train_final.to_csv(f'{output_path}/X_train.csv', index=False)
X_val_final.to_csv(f'{output_path}/X_val.csv', index=False)
X_test_final.to_csv(f'{output_path}/X_test.csv', index=False)

y_train_sm.to_csv(f'{output_path}/y_train.csv', index=False)
y_val.to_csv(f'{output_path}/y_val.csv', index=False)
y_test.to_csv(f'{output_path}/y_test.csv', index=False)